In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pickle

In [2]:
df = pd.read_csv("student_data.csv")

In [3]:
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [4]:
df["Passing Status"] = (df["G3"] >= 10).astype(int)

In [5]:
df.drop(columns=["G3","G2"],axis = 1, inplace = True)

In [6]:
X = df.drop(columns = ["Passing Status"], axis = 1)
y = df["Passing Status"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42, stratify=y)

In [8]:
cat_cols = df.select_dtypes("object").columns.to_list()
transformer = ColumnTransformer(
    transformers = [
        ("onehot",OneHotEncoder(drop="first",handle_unknown="ignore"),cat_cols)
    ], remainder= "passthrough" 
)
X_train = transformer.fit_transform(X_train)
X_test = transformer.transform(X_test)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
logistic = LogisticRegression(random_state=42, class_weight = "balanced")
logistic.fit(X_train_scaled,y_train)
y_pred = logistic.predict(X_test_scaled)
print(accuracy_score(y_test,y_pred))

0.7341772151898734


In [10]:
saved_data = {
    'model': logistic,
    'scaler': scaler,
    'transformer': transformer
}

with open('student_model.pkl', 'wb') as f:
    pickle.dump(saved_data, f)

In [11]:
pd.DataFrame(X_test_scaled).to_csv("testdatascaled.csv", index = False)